In [1]:
import pandas as pd
import numpy as np

In [2]:
customer_df = pd.read_csv("../Data/customer.csv")
orders_df = pd.read_csv("../Data/orders.csv")
products_df = pd.read_csv("../Data/products.csv")

In [119]:
orders_df.isna().sum()

order_id          0
customer_id       0
product_id        0
order_date        0
quantity          0
payment_method    0
dtype: int64

In [38]:
# print(len(customer_df))
customer_df["age"].isna().sum()

np.int64(0)

In [45]:
customer_df["gender"].isna().sum()

np.int64(0)

In [5]:
print(f"Missing values: {products_df.isna().sum()}")

Missing values: product_id      0
product_name    0
category        0
price           0
dtype: int64


In [17]:
# print(len(orders_df))
orders_df["order_id"].unique()

<StringArray>
['O00001', 'O00002', 'O00003', 'O00004', 'O00005', 'O00006', 'O00007',
 'O00008', 'O00009', 'O00010',
 ...
 'O03991', 'O03992', 'O03993', 'O03994', 'O03995', 'O03996', 'O03997',
 'O03998', 'O03999', 'O04000']
Length: 4000, dtype: str

In [15]:
orders_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   order_id        4000 non-null   str  
 1   customer_id     4000 non-null   str  
 2   product_id      4000 non-null   str  
 3   order_date      4000 non-null   str  
 4   quantity        4000 non-null   int64
 5   payment_method  4000 non-null   str  
dtypes: int64(1), str(5)
memory usage: 187.6 KB


In [31]:
# print(len(products_df))
orders_df["quantity"].max()

np.int64(5)

In [12]:
# print(f"{products_df["price"].isna().sum()}")
products_df["price"].sum()

np.float64(11206.630000000001)

In [14]:
products_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 80 entries, 0 to 79
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product_id    80 non-null     str    
 1   product_name  80 non-null     str    
 2   category      80 non-null     str    
 3   price         80 non-null     float64
dtypes: float64(1), str(3)
memory usage: 2.6 KB


In [26]:
products_df["price"].sum()

np.float64(11206.630000000001)

* The Dataset is clean and is ready to get exported to SQL, 


* # Export to the SQLITE

In [39]:
import sqlite3
# Create reatail_sales db
conn = sqlite3.connect('../Data/retail_sales.db')

# Export to SQL and create 3 tables
orders_df.to_sql('orders', conn, if_exists='replace', index=False)
customer_df.to_sql('customers', conn, if_exists='replace', index=False)
products_df.to_sql('products', conn, if_exists='replace', index=False)

print("Data is exported to sql!")

Data is exported to sql!


* Total `Customers` in the `dataset`.

In [52]:
query = """
    select
        count(distinct customer_id) as total_customers
      from customers
"""
pd.read_sql_query(query,conn)

,total_customers
0,1200


* Cities with highest customers.

In [53]:
customer_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   customer_id     1200 non-null   str  
 1   gender          1200 non-null   str  
 2   age             1200 non-null   int64
 3   city            1200 non-null   str  
 4   signup_date     1200 non-null   str  
 5   loyalty_member  1200 non-null   str  
dtypes: int64(1), str(5)
memory usage: 56.4 KB


In [63]:
query = """
select city,
    count(distinct customer_id) as num_ppl
from customers
    group by city
    order by customer_id desc
    limit 15
"""
pd.read_sql_query(query, conn)

,city,num_ppl
0,Birmingham,159
1,Leeds,146
2,Bristol,156
3,Sheffield,163
4,Nottingham,160
5,London,146
6,Manchester,133
7,Liverpool,137


In [65]:
customer_df["city"].unique()

<StringArray>
[ 'Liverpool', 'Manchester',     'London', 'Nottingham',  'Sheffield',
    'Bristol',      'Leeds', 'Birmingham']
Length: 8, dtype: str

* The most purchased products

In [93]:
orders_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   order_id        4000 non-null   str  
 1   customer_id     4000 non-null   str  
 2   product_id      4000 non-null   str  
 3   order_date      4000 non-null   str  
 4   quantity        4000 non-null   int64
 5   payment_method  4000 non-null   str  
dtypes: int64(1), str(5)
memory usage: 187.6 KB


In [83]:
query = """
select 
    p.product_id,
    p.product_name,
    sum(o.quantity) as total_purchased
from orders o
join products p 
on o.product_id = p.product_id
group by p.product_id, p.product_name
order by total_purchased desc
limit 5;
"""
pd.read_sql_query(query, conn)

,product_id,product_name,total_purchased
0,P055,Monitor 55,201
1,P060,Wireless Mouse 60,193
2,P018,Bluetooth Speaker 18,191
3,P065,USB Cable 65,178
4,P067,Face Cream 67,178


* Total `Revenue` Generated.

In [87]:
query = """
select 
    sum(o.quantity * p.price) as total_revenue
from orders o
join products p on o.product_id = p.product_id;
"""
pd.read_sql_query(query, conn)

,total_revenue
0,1644817.71


* Which category generated the most revenue

In [91]:
query = """
select 
    p.category,
    sum(o.quantity * p.price) as total_revenue
from orders o
join products p on o.product_id = p.product_id
group by p.category
order by total_revenue desc
limit 1;
"""
pd.read_sql_query(query, conn)

,category,total_revenue
0,Home,473002.14


* Average order quantity.

In [94]:
query = """
select 
    avg(quantity) as average_orders_quantity
from orders;
"""
pd.read_sql_query(query, conn)

,average_orders_quantity
0,2.97475


* City that generates the most revenue.

* Here we need to join all the tables.

In [96]:
query = """
select 
    c.city,
    sum(o.quantity * p.price) as total_revenue
from orders o
join customers c on o.customer_id = c.customer_id
join products p on o.product_id = p.product_id
group by c.city
order by total_revenue desc;
"""
pd.read_sql_query(query, conn)

,city,total_revenue
0,Sheffield,233607.88
1,Birmingham,224892.95
2,Nottingham,213151.66
3,Bristol,202317.33
4,Liverpool,197613.63
5,Leeds,194316.01
6,London,194041.93
7,Manchester,184876.32


* Top 10 highest spending customers.

In [105]:
query = """
select 
    c.customer_id,
    c.city,
    sum(o.quantity * p.price) as total_spent
from orders o
join customers c on o.customer_id = c.customer_id
join products p on o.product_id = p.product_id
group by c.customer_id,c.city
order by total_spent desc
limit 10;
"""
pd.read_sql_query(query, conn)

,customer_id,city,total_spent
0,C0299,Birmingham,5124.21
1,C0094,Bristol,5078.51
2,C0588,Sheffield,4874.30
3,C0826,Birmingham,4685.22
4,C0334,Liverpool,4515.14
5,C0291,Sheffield,4508.31
6,C0765,Birmingham,4505.37
7,C0157,Manchester,4459.55
8,C1167,Nottingham,4425.53
9,C0674,London,4356.89


* A Month with highest sales.

In [109]:
query = """
select 
    strftime('%Y-%m', o.order_date) as month,
    sum(o.quantity * p.price) as total_sales
from orders o
join products p on o.product_id = p.product_id
group by month
order by total_sales desc
limit 5;
"""
pd.read_sql_query(query, conn)

,month,total_sales
0,2024-01,127940.85
1,2023-01,108265.02
2,2024-02,108166.21
3,2023-06,107246.94
4,2023-03,103306.51


* What is the most common payment method

In [116]:
query = """
select 
    payment_method,
    count(*) as total_order_method
from orders
group by payment_method
order by total_order_method desc
limit 5;
"""
pd.read_sql_query(query, conn)

,payment_method,total_order_method
0,Cash,1362
1,Card,1345
2,Online,1293
